# 特征选择方法概述

## 引言

在机器学习建模中，数据质量至关重要，而特征选择是提升数据质量的核心技术。研究表明，80% 的模型性能取决于数据质量。本文将系统讲解特征选择的三大类方法：过滤法、包装法、嵌入法，帮助初学者理解并掌握这些技术。

## 一、过滤法（Filter Method）

### 1.1 定义与原理

过滤法通过统计指标评估特征与目标变量的相关性，独立于机器学习模型进行筛选。它适用于数据预处理阶段，计算效率高，是特征选择的基础方法。

### 1.2 典型方法

#### (1) 方差过滤

**原理**：去除方差低于阈值的低区分度特征，方差越小说明特征的取值差异越小，对模型的区分能力越弱。

**步骤**：

  1. 计算每个特征的方差。
  2. 设置阈值（如 0.1）。
  3. 删除方差低于阈值的特征。

In [14]:
from sklearn.feature_selection import VarianceThreshold

# 示例数据
X = [[0, 2, 0, 3], [0, 1, 4, 3], [0, 1, 1, 3]]

# 方差过滤
selector = VarianceThreshold(threshold=0.5)
X_new = selector.fit_transform(X)

print("原始特征:\n", X)
print("选择后的特征:\n", X_new)

原始特征:
 [[0, 2, 0, 3], [0, 1, 4, 3], [0, 1, 1, 3]]
选择后的特征:
 [[0]
 [4]
 [1]]


**输出结果解释**：原始特征有 4 个维度，经过方差过滤后，保留了方差较大的特征维度，减少了特征数量，降低了模型的复杂度。

#### (2) 皮尔逊相关系数

**定义**：衡量两个连续变量线性相关程度的统计量，计算公式为：

$$ r = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^n (x_i - \bar{x})^2} \sqrt{\sum_{i=1}^n (y_i - \bar{y})^2}} $$

其中：

- $x_i$, $y_i$ 为变量观测值
- $\bar{x}$, $\bar{y}$ 为变量均值
- $n$ 为样本数量

> [皮尔逊相关系数](https://en.wikipedia.org/wiki/Pearson_correlation_coefficient)

**原理**：皮尔逊相关系数是一种统计指标，用于衡量两个连续变量之间的线性相关性，其取值范围在 - 1 到 1 之间。绝对值越接近 1，相关性越强；绝对值越接近 0，相关性越弱。

**步骤**：

  1. 计算特征与目标的相关系数。
  2. 筛选绝对值大于阈值的特征。

In [16]:
import pandas as pd

# 示例数据
data = {
    'feature1': [1, 2, 3, 4, 5],
    'feature2': [2, 3, 4, 5, 6],
    'feature3': [10, 8, 6, 4, 2],
    'target': [5, 7, 9, 11, 13]
}
df = pd.DataFrame(data)

# 计算相关系数
corr_matrix = df.corr()
target_corr = corr_matrix['target'].abs()

# 筛选相关性大于 0.5 的特征
selected_features = target_corr[target_corr > 0.5].index
print("被选中的特征:", selected_features.tolist())

被选中的特征: ['feature1', 'feature2', 'feature3', 'target']


**输出结果解释**：通过计算特征与目标的皮尔逊相关系数，筛选出与目标相关性较高的特征，这些特征对预测目标变量更有价值。

#### (3) 卡方检验

**定义**：用于检验两个分类变量独立性的假设检验方法，其原假设为"特征与目标变量相互独立"。卡方统计量计算公式：

$$ \chi^2 = \sum_{i=1}^{n} \frac{(O_i - E_i)^2}{E_i} $$

其中：

- $O_i$ 为观察频数（实际样本值）
- $E_i$ 为期望频数（独立假设下的理论值）
- $n$ 为特征取值类别数 × 目标类别数

**原理**：检验分类特征与目标的独立性，卡方值越大，说明特征与目标的关联性越强，相关性也就越强。

**步骤**：

  1. 计算每个特征的卡方统计量。
  2. 按 p 值筛选特征，通常设定一个 p 值阈值（如 0.05），p 值小于阈值的特征被认为与目标相关。

In [17]:
from sklearn.feature_selection import SelectKBest, chi2

# 示例数据
X = [[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 1, 0], [1, 0, 0, 1]]
y = [0, 1, 0, 1]

# 卡方检验
selector = SelectKBest(chi2, k=2)
X_new = selector.fit_transform(X, y)

print("原始特征数量:", len(X[0]))
print("选择后的特征数量:", X_new.shape[1])

原始特征数量: 4
选择后的特征数量: 2


**输出结果解释**：从原始特征中选择与目标相关性较强的 2 个特征，减少了特征维度，提高了模型的效率和性能。

## 二、包装法（Wrapper Method）

### 2.1 定义与原理

包装法通过迭代训练模型评估特征子集效果，根据模型的性能来选择最优的特征组合。这种方法可以获得较好的特征选择效果，但计算成本高，适合特征量小于 20 的场景。

### 2.2 典型方法

#### (1) 递归特征消除（RFE）

**原理**：反向迭代剔除最不重要特征，每次训练模型后，根据特征的重要性权重，剔除权重最小的特征，逐步缩小特征集合，直到达到预定的特征数量。

**步骤**：

  1. 用全部特征训练模型。
  2. 移除权重最小的特征。
  3. 重复直到特征数达标。

In [18]:
from sklearn.feature_selection import RFE
from sklearn.svm import SVC

# 示例数据
X = [[2, 2, 3], [3, 4, 5], [5, 6, 7], [7, 8, 9]]
y = [0, 0, 1, 1]

# RFE
estimator = SVC(kernel="linear")
selector = RFE(estimator, n_features_to_select=2)
X_new = selector.fit_transform(X, y)

print("原始特征数量:", len(X[0]))
print("选择后的特征数量:", X_new.shape[1])
print("被选中的特征索引:", selector.support_)

原始特征数量: 3
选择后的特征数量: 2
被选中的特征索引: [False  True  True]


**输出结果解释**：从原始特征中选择 2 个最重要的特征，通过递归剔除不重要的特征，得到最优的特征组合，提高了模型的性能。

#### (2) 前向选择

**原理**：正向逐步添加提升最大的特征，从空集开始，每次添加一个使模型性能提升最大的特征，直到性能不再显著提升。

**步骤**：

  1. 从空集开始。
  2. 每次添加使模型提升最大的特征。
  3. 直到性能不再显著提升。

In [19]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector

# 示例数据
X = [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]]
y = [3, 6, 9, 12]

# 前向选择，使用 2 折交叉验证
selector = SequentialFeatureSelector(LinearRegression(), n_features_to_select=2, direction="forward", cv=2)
X_new = selector.fit_transform(X, y)

print("原始特征数量:", len(X[0]))
print("选择后的特征数量:", X_new.shape[1])
print("被选中的特征索引:", selector.get_support(indices=True))

原始特征数量: 3
选择后的特征数量: 2
被选中的特征索引: [0 1]


**输出结果解释**：通过前向选择，逐步添加对模型性能提升最大的特征，最终得到 2 个最优特征，提高了模型的预测能力。

## 三、嵌入法（Embedded Method）

### 3.1 定义与原理

嵌入法将特征选择融入模型训练过程，模型在训练过程中自动学习特征的重要性，并进行特征选择。这种方法兼具效率与效果，但模型选择受限，不同的模型可能有不同的特征选择方式。

### 3.2 典型方法

#### (1) L1 正则化（Lasso）

**原理**：通过系数压缩实现特征稀疏化，`L1` 正则化会将一些特征的系数压缩为 0，从而实现特征选择。

In [20]:
from sklearn.linear_model import Lasso
import numpy as np

# 示例数据
X = [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]]
y = [3, 6, 9, 12]

# Lasso 回归
lasso = Lasso(alpha=0.1)
lasso.fit(X, y)

print("特征系数:", lasso.coef_)
print("被选中的特征索引:", np.where(lasso.coef_ != 0)[0])

特征系数: [0.99111111 0.         0.        ]
被选中的特征索引: [0]


**输出结果解释**：`Lasso` 回归通过 `L1` 正则化，将一些特征的系数压缩为 0，剩下的特征系数不为 0 的特征即为被选中的特征，实现了特征选择。

#### (2) 树模型特征重要性

**原理**：基于分裂时的信息增益评估特征重要性，信息增益越大，说明特征对划分数据集的能力越强，特征越重要。

In [21]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# 示例数据
X = [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]]
y = [3, 6, 9, 12]

# 随机森林回归
rf = RandomForestRegressor(n_estimators=100)
rf.fit(X,y)

print("特征重要性:", rf.feature_importances_)
print("被选中的特征索引:", np.argsort(rf.feature_importances_)[-2:])

特征重要性: [0.29601527 0.39270876 0.31127597]
被选中的特征索引: [2 1]


**输出结果解释**：随机森林模型通过计算每个特征的重要性，根据特征的重要性进行排序，选择重要性较高的特征，用于模型的预测。

## 四、方法对比与实践指南

| **方法类型** | **计算速度** | **交互性考虑** | **典型场景** |
|----------|----------|------------|----------|
| 过滤法   | ★★★      | ×          | 数据预处理 |
| 包装法   | ★        | ✓          | 特征量 < 20 |
| 嵌入法   | ★★       | ✓          | 模型调优 |

**最佳实践路线图**：

1. **方差过滤**：初步去除低区分度特征。
2. **卡方 / MIC 初筛**：进一步筛选与目标相关性较强的特征。
3. **Lasso / 树模型**：利用模型的特性进行特征选择，优化特征集合。
4. **RFE 精调**：对特征进行精细调整，得到最优的特征组合。

## 五、总结

特征选择是建模流程中的“特征降噪”过程。理解不同方法的数学原理（如卡方检验的 $ \chi^2 $ 统计量、Lasso 的 $ L1 $ 范数）是灵活运用的关键。